# 1-4 RAG 이론과 핵심 모듈

원본 슬라이드: [1-4-rag-theory-and-core-modules.pdf](./1-4-rag-theory-and-core-modules.pdf)  
이 노트북은 슬라이드의 개념 순서를 따라가는 학습용 해설과 작은 실험입니다. 원본 PDF는 그대로 보존합니다.

## 학습 목표
- RAG와 Fine-Tuning의 용도 차이를 설명한다.
- 로딩→분할→임베딩→저장→검색→생성의 단계를 추적한다.
- 청킹과 검색 결과가 답변 근거에 미치는 영향을 검토한다.

**실습 방식:** 예제를 먼저 실행해 관찰하고, CodeMate에 개선을 요청한 뒤 생성 결과를 직접 검토합니다. 실제 CodeMate 메뉴·버튼 이름은 강의장 환경에서 확인하세요.


## 슬라이드별 찾아보기

각 쪽의 소제목과 첫 핵심 문구를 찾아가기 위한 표입니다. 본문의 설명·실습과 함께 읽고, 원본의 그림·도표·화면 예시는 쪽 번호를 눌러 확인하세요. PDF에서 수식 또는 그림 안 글자가 추출되지 않는 쪽은 원본 화면을 기준으로 읽습니다.

| PDF 쪽 | 소제목 | 확인할 내용 |
|---|---|---|
| [1쪽](./1-4-rag-theory-and-core-modules.pdf#page=1) | 1-4 | RAG 이론 & 실습 |
| [2쪽](./1-4-rag-theory-and-core-modules.pdf#page=2) | Retrieval Augmented Generation | Fine-Tuning과의 비교 |
| [3쪽](./1-4-rag-theory-and-core-modules.pdf#page=3) | Retrieval Augmented Generation | 원본 슬라이드의 그림·배치 확인 |
| [4쪽](./1-4-rag-theory-and-core-modules.pdf#page=4) | Fine-Tuning과의 비교 (1) | RAG: 모델이 모르는 외부 지식을 검색해서 답변 생성에 함께 넣어주는 방식 |
| [5쪽](./1-4-rag-theory-and-core-modules.pdf#page=5) | Fine-Tuning과의 비교 (2) | Fine-Tuning과 RAG는 해결하려는 문제가 다르다. |
| [6쪽](./1-4-rag-theory-and-core-modules.pdf#page=6) | Naive -> Advanced -> Modular RAG 발전 과정 (1) | Naive RAG는 RAG의 가장 기본적인 형태 |
| [7쪽](./1-4-rag-theory-and-core-modules.pdf#page=7) | Naive -> Advanced -> Modular RAG 발전 과정 (2) | Advanced RAG는 검색 품질과 답변 품질을 높이기 위한 보완 단계를 추가한 형태 |
| [8쪽](./1-4-rag-theory-and-core-modules.pdf#page=8) | Naive -> Advanced -> Modular RAG 발전 과정 (3) | Modular RAG는 RAG 단계를 독립적인 모듈처럼 조합하는 방식 |
| [9쪽](./1-4-rag-theory-and-core-modules.pdf#page=9) | RAG 핵심 모듈 | 원본 슬라이드의 그림·배치 확인 |
| [10쪽](./1-4-rag-theory-and-core-modules.pdf#page=10) | Loading, Splitting, Embedding (1) | Loading: 원천 데이터를 RAG 시스템이 다룰 수 있는 문서 형태로 불러오는 단계 |
| [11쪽](./1-4-rag-theory-and-core-modules.pdf#page=11) | Loading, Splitting, Embedding (2) | Splitting: 긴 문서를 검색하기 좋은 작은 단위로 나누는 단계 |
| [12쪽](./1-4-rag-theory-and-core-modules.pdf#page=12) | Loading, Splitting, Embedding (3) | Embedding: 텍스트의 의미를 벡터로 변환하는 단계 |
| [13쪽](./1-4-rag-theory-and-core-modules.pdf#page=13) | Storage, Retrieval, Generation (1) | Storage: 임베딩된 청크와 원문, 메타데이터를 저장하는 단계 |
| [14쪽](./1-4-rag-theory-and-core-modules.pdf#page=14) | Storage, Retrieval, Generation (2) | Retrieval: 사용자의 질문과 관련 있는 문서 청크를 찾는 단계 |
| [15쪽](./1-4-rag-theory-and-core-modules.pdf#page=15) | Storage, Retrieval, Generation (3) | Generation: 검색된 근거를 바탕으로 최종 답변을 생성하는 단계 |
| [16쪽](./1-4-rag-theory-and-core-modules.pdf#page=16) | 청킹 전략과 임베딩 | 원본 슬라이드의 그림·배치 확인 |
| [17쪽](./1-4-rag-theory-and-core-modules.pdf#page=17) | Fixed-size, Recursive, Semantic Chunking (1) | Fixed-size Chunking: 글자 수나 토큰 수 기준으로 일정하게 나누는 방식 |
| [18쪽](./1-4-rag-theory-and-core-modules.pdf#page=18) | Fixed-size, Recursive, Semantic Chunking (2) | Semantic Chunking: 의미 변화 지점을 기준으로 나누는 방식 |
| [19쪽](./1-4-rag-theory-and-core-modules.pdf#page=19) | 임베딩 모델 선택 기준 | 임베딩 모델은 문장 의미를 벡터로 얼마나 잘 표현하는지를 결정 |


## 1. RAG가 해결하는 문제 (슬라이드 3–5)

RAG는 질문과 관련된 외부 자료를 **검색해 현재 요청의 맥락으로 제공**합니다. 사내 문서나 갱신된 자료를 참조할 때 유용합니다. Fine-Tuning은 모델의 말투·형식·업무 스타일 조정에 더 가깝습니다. 검색 자료에 답이 없으면 답을 꾸미지 않도록 지시하고 결과도 검토해야 합니다.

## 2. 기본 흐름과 확장 (슬라이드 6–8)

Naive RAG: 로딩 → 청크 분할 → 임베딩·저장 → 질문 검색 → 검색 근거로 답변. Advanced RAG는 질문 재작성, 키워드·벡터 혼합 검색, 재정렬, 근거 일치 점검을 추가할 수 있습니다. Modular RAG는 이 단계들을 교체 가능한 모듈로 조합합니다. 먼저 어느 단계가 실패했는지 분리해서 살피세요.

### 슬라이드의 실제 구성요소 연결 (슬라이드 10–15)

원본 코드는 `PDFMinerLoader` → `RecursiveCharacterTextSplitter` → `OpenAIEmbeddings` → `Chroma` → `as_retriever(k=3)` 순서입니다. 아래는 이름과 데이터 이동을 확인하기 위한 읽기용 뼈대입니다. 실제 실행에는 문서 파일·설치 패키지·승인된 모델 연결이 필요합니다. 사내망에서는 강의장 provider 설정을 우선 확인하세요.

```python
# documents = PDFMinerLoader("./docs/manual.pdf").load()
# chunks = RecursiveCharacterTextSplitter(
#     chunk_size=800, chunk_overlap=120
# ).split_documents(documents)
# embeddings = 승인된 임베딩 모델
# vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
# docs = retriever.invoke("Agent에서 메모리는 왜 필요한가?")
# print(docs[0].metadata, docs[0].page_content[:200])
```

**확인할 연결:** `documents`와 `chunks`의 출처 메타데이터가 `docs`까지 남는지, `k=3`이 충분한지, 검색된 문서 안에 답이 실제로 있는지 확인하세요.

## 3. Loading, Splitting, Embedding (슬라이드 9–12)

로딩 시 본문과 함께 파일명·페이지 같은 출처를 보존합니다. 청크가 너무 크면 필요한 부분을 찾기 어렵고, 너무 작으면 문맥이 잘립니다. 임베딩은 문장을 벡터로 나타내 의미 기반 검색에 활용합니다. 아래는 외부 임베딩 서비스 없이 **단어 겹침으로 검색 단계를 흉내 내는 실험**입니다. 벡터 검색과 같지 않습니다.

In [ ]:
documents = [
    {"source": "guide.pdf", "page": 2, "text": "RAG는 질문과 관련된 문서를 검색한다"},
    {"source": "guide.pdf", "page": 5, "text": "청크는 긴 문서를 작은 단위로 나눈 것이다"},
    {"source": "faq.pdf", "page": 1, "text": "답변에는 검색 근거의 출처를 표시한다"},
]
question = "RAG 문서 검색"
terms = set(question.split())
ranked = sorted(documents, key=lambda d: len(terms & set(d["text"].split())), reverse=True)
for doc in ranked:
    print(f"{doc['source']} p.{doc['page']} | {doc['text']}")

**관찰:** 한국어 조사 때문에 단순 단어 겹침은 어떤 문서를 놓칠 수 있나요? 임베딩 검색을 도입하더라도 “정답 문서가 상위에 있는지”를 실제 질문으로 평가해야 합니다.

## 4. Storage, Retrieval, Generation (슬라이드 13–15)

저장소에는 청크·임베딩·메타데이터가 함께 필요합니다. 검색 단계의 `k` 값, 유사도 기준, 필터는 가져오는 근거를 바꿉니다. 생성 단계에는 질문과 **선택된 근거**를 함께 제공하고, 근거 밖의 추측을 막고 출처를 표시합니다. 검색이 틀렸다면 유창한 답변도 믿기 어렵습니다.

In [ ]:
selected = ranked[:2]
context = "\n".join(f"[{d['source']} p.{d['page']}] {d['text']}" for d in selected)
prompt = f"질문: {question}\n근거:\n{context}\n근거에서 답을 찾을 수 없으면 확인할 수 없다고 답하세요."
print(prompt)

## 5. 청킹과 임베딩 모델 선택 (슬라이드 16–19)

**Fixed-size**는 일정 길이로 자르지만 문장 중간이 끊길 수 있습니다. **Recursive**는 문단·문장 등 자연스러운 경계를 찾습니다. **Semantic**은 주제 변화에 맞춰 나누지만 계산 비용이 큽니다. 한국어와 도메인 용어가 있는 자료에서는 실제 질문·기대 문서·검색 순위·비용·속도를 함께 확인하세요.

In [ ]:
text = "RAG는 문서를 검색합니다. 검색 결과는 질문과 관련되어야 합니다. 답변에는 출처가 필요합니다."
chunks = [part.strip() for part in text.split(". ") if part.strip()]
for i, chunk in enumerate(chunks, 1):
    print(f"청크 {i}: {chunk}")

### CodeMate 페어 실습

> 위 문서 목록에 `retrieve(question, documents, k)` 함수를 추가해 주세요. 검색 결과가 없으면 빈 목록을 반환하고, 반환값에는 출처와 페이지를 보존해 주세요. 먼저 테스트 질문과 기대 문서 3쌍을 정의한 뒤 구현해 주세요.

**리뷰 질문:** 근거가 없는 질문에 답을 만들어 내지 않나요? 출처가 끝까지 유지되나요? 청크 길이를 바꾸면 검색 결과가 어떻게 변하나요?

## 정리

1. 최신 사내 규정 확인에는 RAG와 Fine-Tuning 중 무엇이 직접적인가요?
2. 생성 단계만 고치면 잘못된 검색 결과를 해결할 수 있나요?
3. 검색 품질을 확인할 질문·정답 문서 쌍을 하나 적어 보세요.